# 03 — Fusion & Report

Blueprint for Module 4. Input: `{region}_buildings.csv` (Module 2/3) + `{region}_rivers.geojson`
(Module 1). Turns "a structure" into "an encroachment": confirm each building's true distance
to a river line, apply the RF material-overlap rule, and calibrate the flagging distance
against the one region with real ground truth (Kasarani). Output:
`{region}_encroaching_buildings.csv` and `pipeline_summary.json`.

In [6]:
import json
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

DATA_DIR = '../data/processed'
REGIONS = ['Kasarani', 'Gatharaini', 'Motoine']
CONFIDENCE_THRESHOLD = 0.7
MATERIAL_OVERLAP_THRESHOLD = 0.05  # flag if RF built-up probability >= 5%

## Step 1 — True distance to river, not just "inside the buffer"

Module 1's buffer polygon is a coarse 60m cutoff; calibration below needs exact metre
distances well under that, so this measures straight-line distance to the nearest
river/stream line for every building, in the metric CRS.

In [7]:
def add_river_distance(buildings_df, region_key):
    rivers = gpd.read_file(f'{DATA_DIR}/{region_key.lower()}_rivers.geojson')  # already EPSG:32737
    river_union = rivers.union_all()

    points = gpd.GeoDataFrame(
        buildings_df.copy(),
        geometry=[Point(xy) for xy in zip(buildings_df['lon'], buildings_df['lat'])],
        crs='EPSG:4326',
    ).to_crs(epsg=32737)

    points['distance_to_river_m'] = points.geometry.distance(river_union)
    return pd.DataFrame(points.drop(columns='geometry'))

## Step 2 — Apply confidence + material-overlap filters

Two independent checks per building before it counts as a real candidate: Open Buildings'
own confidence score, and the RF material-overlap rule (flag only if the RF built-up
probability at that point clears `MATERIAL_OVERLAP_THRESHOLD`).

In [8]:
def filter_candidates(buildings_df):
    mask = (
        (buildings_df['confidence'] >= CONFIDENCE_THRESHOLD)
        & (buildings_df['rf_builtup_prob'] >= MATERIAL_OVERLAP_THRESHOLD)
    )
    return buildings_df[mask].copy()

## Step 3 — Calibrate the flagging distance against real ground truth

**Only Kasarani has an independent number to check against** — Pamoja Trust's manual survey,
~700 buildings. Sweep candidate distances and keep whichever reproduces that count most
closely; apply the same distance to Gatharaini and Motoine, but say plainly that those two are
an untested extrapolation, not a second calibration — there's no field survey for them.

In [9]:
PAMOJA_TRUST_KASARANI_COUNT = 700


def calibrate_distance(candidates_df, target_count, sweep_range=range(10, 61)):
    sweep = {d: int((candidates_df['distance_to_river_m'] <= d).sum()) for d in sweep_range}
    best_d = min(sweep, key=lambda d: abs(sweep[d] - target_count))
    return best_d, sweep

## Step 4 — Run for all three regions, save the triage tables

In [13]:
from pathlib import Path

# Replace with the path to the folder where your CSV files are saved
DATA_DIR = Path(r"C:\Users\HP\Downloads")  # Example path

In [14]:
summary = {}
calibrated_distance_m = None

for region in REGIONS:
    print(f'--- {region} ---')
    buildings = pd.read_csv(f'{DATA_DIR}/{region.lower()}_buildings.csv')
    with_distance = add_river_distance(buildings, region)
    candidates = filter_candidates(with_distance)

    if region == 'Kasarani':
        calibrated_distance_m, sweep = calibrate_distance(candidates, PAMOJA_TRUST_KASARANI_COUNT)
        print(f'Calibrated distance (Kasarani, vs. Pamoja Trust ~{PAMOJA_TRUST_KASARANI_COUNT}): '
              f'{calibrated_distance_m}m -> {sweep[calibrated_distance_m]} buildings')

    encroaching = candidates[candidates['distance_to_river_m'] <= calibrated_distance_m]
    out_path = f'{DATA_DIR}/{region.lower()}_encroaching_buildings.csv'
    encroaching.to_csv(out_path, index=False)

    summary[region] = {
        'buildings_screened': len(with_distance),
        'passed_confidence_and_material_filter': len(candidates),
        'encroaching_count': int(len(encroaching)),
        'calibrated_against_ground_truth': region == 'Kasarani',
    }
    print(summary[region])

summary['calibrated_distance_m'] = calibrated_distance_m
summary['note'] = (
    'Kasarani is the only region calibrated against an independent field count '
    f'(Pamoja Trust, ~{PAMOJA_TRUST_KASARANI_COUNT}). Gatharaini and Motoine reuse that same '
    f'{calibrated_distance_m}m cutoff untested — no ground truth exists yet for either.'
)

with open(f'{DATA_DIR}/pipeline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

pd.DataFrame({k: v for k, v in summary.items() if k in REGIONS}).T

--- Kasarani ---
Calibrated distance (Kasarani, vs. Pamoja Trust ~700): 16m -> 731 buildings
{'buildings_screened': 56678, 'passed_confidence_and_material_filter': 56415, 'encroaching_count': 731, 'calibrated_against_ground_truth': True}
--- Gatharaini ---
{'buildings_screened': 27123, 'passed_confidence_and_material_filter': 27046, 'encroaching_count': 87, 'calibrated_against_ground_truth': False}
--- Motoine ---
{'buildings_screened': 28554, 'passed_confidence_and_material_filter': 28104, 'encroaching_count': 348, 'calibrated_against_ground_truth': False}


,buildings_screened,passed_confidence_and_material_filter,encroaching_count,calibrated_against_ground_truth
Kasarani,56678,56415,731,True
Gatharaini,27123,27046,87,False
Motoine,28554,28104,348,False


## POSTPROCESSING

## 1. Export Encroaching Buildings to GeoJSON
Convert the output CSV files into spatial GeoJSON layers so they can be rendered in GIS software or web maps.

In [15]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point

# Match the directory path where cell 13 saved the outputs
DATA_DIR = Path(r"C:\Users\HP\Downloads")
REGIONS = ['kasarani', 'gatharaini', 'motoine']

for region in REGIONS:
    csv_path = DATA_DIR / f"{region}_encroaching_buildings.csv"
    df = pd.read_csv(csv_path)
    
    gdf = gpd.GeoDataFrame(
        df, 
        geometry=[Point(xy) for xy in zip(df['lon'], df['lat'])], 
        crs='EPSG:4326'
    )
    
    geojson_path = DATA_DIR / f"{region}_encroaching_buildings.geojson"
    gdf.to_file(geojson_path, driver='GeoJSON')
    print(f"Exported {region} to {geojson_path}")

Exported kasarani to C:\Users\HP\Downloads\kasarani_encroaching_buildings.geojson
Exported gatharaini to C:\Users\HP\Downloads\gatharaini_encroaching_buildings.geojson
Exported motoine to C:\Users\HP\Downloads\motoine_encroaching_buildings.geojson


## 2. Generate Interactive Maps (Folium)
Create an interactive HTML map visualizing river lines, buffer boundaries, and flagged encroaching structures.

In [16]:
import folium
import geopandas as gpd
from pathlib import Path

# Set the path to your downloads folder
DATA_DIR = Path(r"C:\Users\HP\Downloads")

# Load layers for Kasarani using Path objects
rivers = gpd.read_file(DATA_DIR / 'kasarani_rivers.geojson')
encroaching = gpd.read_file(DATA_DIR / 'kasarani_encroaching_buildings.geojson')

# Center map on average coordinates
m = folium.Map(location=[encroaching['lat'].mean(), encroaching['lon'].mean()], zoom_start=14)

# Add rivers and encroaching points
folium.GeoJson(rivers, name='River Vector', style_function=lambda x: {'color': 'blue', 'weight': 3}).add_to(m)

for _, row in encroaching.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        color='red',
        fill=True,
        fill_opacity=0.7,
        popup=f"Distance to river: {row['distance_to_river_m']:.1f}m"
    ).add_to(m)

# Save the map to Downloads
out_html = DATA_DIR / 'kasarani_encroachment_map.html'
m.save(out_html)
print(f"Map saved successfully to {out_html}")

Map saved successfully to C:\Users\HP\Downloads\kasarani_encroachment_map.html
